# Pilot scene feasibility and water-level gate

**Status:** this historical feasibility gate is complete. The later date-design notebook proved that 14 scenes are required once mission-specific water patterns are included and froze Design 1 on 2026-08-08.

This notebook records the **scene-feasibility and water-level evidence gate** for the lean Landsat pilot. It tests whether an eight-scene design is feasible and validates the frozen scene–sector water-level table before any dates are selected, imagery is retrieved or shorelines are extracted.

The approved working design is two unique scenes from each of Landsat 5, 7, 8 and 9. Across the eight scenes, every decade and meteorological season must be represented. The check below deliberately applies a strict spatial condition: a scene is counted only if the availability manifest contains it in every ROI intersecting every approved core sector. This is conservative metadata evidence, not proof of usable local pixels.

## Decision boundaries

This stage supports implementation-plan items P012, P020, P023, P024, P025, P026, P043, P044, P055 and P061. It may establish that the design is feasible, verify the water-level evidence and identify LiDAR timing opportunities. It must not:

- rank or select acquisition dates;
- treat scene-wide cloud metadata as local visual quality;
- use plausible shoreline behaviour or erosion rates;
- download imagery or begin extraction; or
- choose the final water-level filter or pilot dates without explicit human review.

The existing pilot design summary is retained as provenance. This notebook applies the subsequently approved eight-scene working design without silently rewriting that source file.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root():
    """Find the repository whether Jupyter started in root or notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "implementation-plan.json").is_file():
            return candidate
    raise FileNotFoundError("Could not locate implementation-plan.json")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from holderness.gtsm import sha256_file
from holderness.pilot import (
    P023_WATER_LEVEL_BANDS,
    PILOT_STILL_WATER_COMPONENT_COLUMNS,
    add_fes_astronomical_tide,
    add_p023_water_bands,
    add_pilot_still_water_anomaly,
    add_sector_coverage,
    collapse_scene_coverage,
)

print(f"Repository root: {REPO_ROOT}")

## 1. Load the recorded evidence

The availability manifest remains the scene authority. Sector and LiDAR tables add local context; neither changes the source scene metadata. The frozen evaluation pool records which eligible scenes received water-level evaluation. Its three derived water-level evidence files are handled together later as an explicit gate.

In [ ]:
paths = {
    "plan": REPO_ROOT / "implementation-plan.json",
    "availability": REPO_ROOT / "data/derived/availability/landsat-scene-availability.csv",
    "sectors": REPO_ROOT / "data/interim/pilot/pilot-sector-candidates.csv",
    "lidar_sector_surveys": REPO_ROOT / "data/interim/validation/ea-lidar-pilot-sector-surveys.csv",
    "lidar_opportunities": REPO_ROOT / "data/interim/validation/landsat-lidar-pilot-opportunities.csv",
    "design_summary": REPO_ROOT / "data/derived/pilot/pilot-design-summary.json",
    "evaluation_pool": REPO_ROOT / "data/derived/pilot/pilot-water-level-evaluation-scenes.csv",
    "scene_water_levels": REPO_ROOT / "data/derived/pilot/pilot-scene-water-levels.csv",
    "water_level_summary": REPO_ROOT / "data/derived/pilot/pilot-water-level-summary.json",
    "msl_merge_audit": REPO_ROOT / "data/derived/pilot/gtsm-msl-merge-audit.json",
}

water_evidence_names = {"scene_water_levels", "water_level_summary", "msl_merge_audit"}
required_now = {name: path for name, path in paths.items() if name not in water_evidence_names}
missing = [str(path.relative_to(REPO_ROOT)) for path in required_now.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required pilot inputs: {missing}")

plan = json.loads(paths["plan"].read_text())
design_summary = json.loads(paths["design_summary"].read_text())
availability = pd.read_csv(paths["availability"])
sectors = pd.read_csv(paths["sectors"])
lidar_sector_surveys = pd.read_csv(paths["lidar_sector_surveys"])
lidar_opportunities = pd.read_csv(paths["lidar_opportunities"])
evaluation_pool = pd.read_csv(paths["evaluation_pool"])

decision_ids = ["P012", "P020", "P023", "P024", "P025", "P026", "P043", "P044", "P055", "P061"]
plan_by_id = {decision.get("id"): decision for decision in plan}
if not set(decision_ids).issubset(plan_by_id):
    raise ValueError("One or more required implementation-plan decisions are missing")

display(
    pd.DataFrame([plan_by_id[decision_id] for decision_id in decision_ids])
    .loc[:, ["id", "phase", "kind", "item"]]
)

In [ ]:
expected_missions = ["L5", "L7", "L8", "L9"]
core_sectors = sectors.loc[sectors["role"].eq("core_candidate")].copy()
backup_sectors = sectors.loc[sectors["role"].eq("backup_candidate")].copy()
core_sector_ids = set(core_sectors["pilot_sector_id"])

assert set(availability["stream"]) == {"landsat"}
assert set(expected_missions).issubset(availability["sensor"].unique())
assert not availability.duplicated(["source_scene_id", "roi_id"]).any()
assert availability["source_scene_id"].notna().all()
assert len(core_sectors) == 3 and len(backup_sectors) == 1
assert core_sectors["review_status"].str.startswith("approved").all()
assert backup_sectors["review_status"].str.startswith("approved").all()
assert set(lidar_opportunities["pilot_sector_id"]) == core_sector_ids
assert not lidar_opportunities.duplicated(["source_scene_id", "pilot_sector_id"]).any()
assert evaluation_pool["source_scene_id"].is_unique
assert evaluation_pool["acquisition_time_utc"].notna().all()

input_record = pd.DataFrame(
    [
        {
            "input": name,
            "repository_path": str(path.relative_to(REPO_ROOT)),
            "records": len(frame),
        }
        for name, path, frame in [
            ("Landsat availability", paths["availability"], availability),
            ("approved sector record", paths["sectors"], sectors),
            ("EA LiDAR sector surveys", paths["lidar_sector_surveys"], lidar_sector_surveys),
            ("Landsat–LiDAR opportunities", paths["lidar_opportunities"], lidar_opportunities),
            ("frozen water-level evaluation pool", paths["evaluation_pool"], evaluation_pool),
        ]
    ]
)

print(f"Availability retrieved at: {availability['metadata_retrieved_at_utc'].nunique()} recorded UTC timestamp(s)")
print(f"Recorded design-summary status: {design_summary['status']}")
display(input_record)
display(
    sectors[["pilot_sector_id", "role", "context_requirement", "review_status", "intersecting_roi_ids"]]
)

## 2. Collapse ROI repetitions and attach conservative sector coverage

A single Landsat acquisition can appear once for each intersected availability ROI. First, invariant metadata are checked and those rows are collapsed to one source scene. A scene covers a sector only when it appears in **all** ROIs named for that sector.

For this feasibility test, a usable scene must also be marked `primary_geometry_eligible` and cover all three core sectors. The Hornsea south-end sector remains a backup and is not an additional requirement. The resulting eligible catalogue is then restricted to the frozen 110-scene water-blind evaluation pool so every scene counted later has complete water-level evidence.

In [ ]:
scene_catalog = collapse_scene_coverage(availability)
scene_catalog = add_sector_coverage(
    scene_catalog,
    sectors,
    required_roles=("core_candidate",),
)

eligible_scenes = scene_catalog.loc[
    scene_catalog["primary_geometry_eligible"].eq(True)
    & scene_catalog["covers_all_required_sectors"].eq(True)
].copy()

assert eligible_scenes["source_scene_id"].is_unique
assert eligible_scenes["covered_sector_ids"].map(
    lambda value: core_sector_ids.issubset(set(value.split("|")))
).all()

eligible_by_id = eligible_scenes.set_index("source_scene_id")
pool_ids = set(evaluation_pool["source_scene_id"])
unknown_pool_ids = pool_ids.difference(eligible_by_id.index)
if unknown_pool_ids:
    raise ValueError(f"Frozen evaluation pool contains ineligible scenes: {sorted(unknown_pool_ids)[:5]}")
pool_utc = pd.to_datetime(evaluation_pool["acquisition_time_utc"], format="mixed", utc=True, errors="coerce")
catalogue_utc = pd.to_datetime(
    evaluation_pool["source_scene_id"].map(eligible_by_id["acquisition_time_utc"]),
    format="mixed",
    utc=True,
    errors="coerce",
)
if pool_utc.isna().any() or catalogue_utc.isna().any() or not np.array_equal(pool_utc.array.asi8, catalogue_utc.array.asi8):
    raise ValueError("Frozen evaluation-pool UTCs must exactly match the availability manifest")
strict_scenes = eligible_scenes.loc[eligible_scenes["source_scene_id"].isin(pool_ids)].copy()
if len(strict_scenes) != len(evaluation_pool):
    raise ValueError("Frozen evaluation pool did not map one-to-one to eligible scenes")

coverage_summary = pd.DataFrame(
    {
        "stage": [
            "manifest ROI rows",
            "unique source scenes",
            "geometry-eligible source scenes",
            "strict metadata-eligible scenes covering every core sector",
            "frozen water-level evaluation scenes",
        ],
        "count": [
            len(availability),
            len(scene_catalog),
            int(scene_catalog["primary_geometry_eligible"].eq(True).sum()),
            len(eligible_scenes),
            len(strict_scenes),
        ],
    }
)
display(coverage_summary)

## 3. Keep LiDAR timing as validation evidence

Environment Agency LiDAR is not used to choose attractive shoreline behaviour. The table below only records potential temporal matches under P061: same-day or ±3 days for primary validation, and ±7 days for sensitivity. Opportunities remain sector-specific even though the scene catalogue is unique by acquisition.

In [ ]:
strict_ids = strict_scenes[["source_scene_id", "sensor"]]
strict_lidar_opportunities = lidar_opportunities.merge(
    strict_ids,
    on=["source_scene_id", "sensor"],
    how="inner",
    validate="many_to_one",
)

lidar_by_scene = (
    strict_lidar_opportunities.groupby("source_scene_id", as_index=False)
    .agg(
        lidar_core_sector_count=("pilot_sector_id", "nunique"),
        nearest_lidar_distance_days=("lidar_distance_days", "min"),
        any_lidar_within_3_days=("lidar_within_3_days", "any"),
        any_lidar_within_7_days=("lidar_within_7_days", "any"),
    )
)
strict_scenes = strict_scenes.merge(
    lidar_by_scene,
    on="source_scene_id",
    how="left",
    validate="one_to_one",
)
strict_scenes[["any_lidar_within_3_days", "any_lidar_within_7_days"]] = (
    strict_scenes[["any_lidar_within_3_days", "any_lidar_within_7_days"]].fillna(False)
)

lidar_evidence = (
    strict_lidar_opportunities.groupby("sensor")
    .agg(
        eligible_unique_scenes=("source_scene_id", "nunique"),
        scene_sector_opportunities=("pilot_sector_id", "size"),
        primary_pairs_within_3_days=("lidar_within_3_days", "sum"),
        sensitivity_pairs_within_7_days=("lidar_within_7_days", "sum"),
    )
    .reindex(expected_missions, fill_value=0)
)
display(lidar_evidence)

## 4. Test the eight-scene design within the evaluated pool

The following tables show the number of frozen, water-level-evaluated scenes by mission, decade and season. Counts are evidence of choice, not a ranking. Cloud cover remains provider scene-wide metadata and will require local visual review only after imagery retrieval is separately authorised.

In [ ]:
decades = ["1990s", "2000s", "2010s", "2020s"]
seasons = ["DJF", "MAM", "JJA", "SON"]
required_per_mission = 2
target_unique_scenes = required_per_mission * len(expected_missions)

mission_feasibility = (
    strict_scenes.groupby("sensor")
    .agg(
        eligible_unique_scenes=("source_scene_id", "nunique"),
        represented_decades=("decade", "nunique"),
        represented_seasons=("season", "nunique"),
        scenes_with_primary_lidar_opportunity=("any_lidar_within_3_days", "sum"),
    )
    .reindex(expected_missions, fill_value=0)
)
mission_feasibility.insert(0, "required_unique_scenes", required_per_mission)
mission_feasibility["mission_count_feasible"] = (
    mission_feasibility["eligible_unique_scenes"] >= required_per_mission
)

mission_by_decade = pd.crosstab(strict_scenes["sensor"], strict_scenes["decade"]).reindex(
    index=expected_missions, columns=decades, fill_value=0
)
mission_by_season = pd.crosstab(strict_scenes["sensor"], strict_scenes["season"]).reindex(
    index=expected_missions, columns=seasons, fill_value=0
)

display(mission_feasibility)
display(Markdown("**Strict eligible scenes by mission and decade**"))
display(mission_by_decade)
display(Markdown("**Strict eligible scenes by mission and season**"))
display(mission_by_season)

In [ ]:
def annotate_count_matrix(ax, table, title):
    image = ax.imshow(table.to_numpy(), cmap="Blues", aspect="auto")
    ax.set_title(title)
    ax.set_xticks(range(len(table.columns)), table.columns)
    ax.set_yticks(range(len(table.index)), table.index)
    ax.set_xlabel("stratum")
    ax.set_ylabel("Landsat mission")
    threshold = table.to_numpy().max() / 2
    for row in range(len(table.index)):
        for column in range(len(table.columns)):
            value = int(table.iloc[row, column])
            colour = "white" if value > threshold else "black"
            ax.text(column, row, value, ha="center", va="center", color=colour)
    return image


figure_path = REPO_ROOT / "outputs/figures/pilot-scene-metadata-feasibility.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
annotate_count_matrix(axes[0], mission_by_decade, "By decade")
annotate_count_matrix(axes[1], mission_by_season, "By meteorological season")
fig.suptitle(
    "Frozen water-level evaluation pool across Landsat strata",
    fontsize=13,
)
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Wrote {figure_path.relative_to(REPO_ROOT)}")

### Joint feasibility across decades and seasons

Separate count tables do not prove that one eight-scene design can cover all required strata at once. The next cell therefore combines **stratum counts only**. For each mission it records the decade/season coverage masks possible with two scenes, then tests whether the four mission-level masks can jointly cover every decade and season. It retains no scene IDs, dates or back-pointers, so it cannot become an accidental shortlist.

In [ ]:
def two_scene_coverage_masks(scene_strata):
    """Return possible decade/season masks using counts, not scene IDs."""
    counts = scene_strata.groupby(["decade", "season"]).size()
    cells = list(counts.items())
    masks = set()
    for first, ((decade_a, season_a), count_a) in enumerate(cells):
        for second in range(first, len(cells)):
            (decade_b, season_b), _ = cells[second]
            if first == second and count_a < 2:
                continue
            decade_mask = (1 << decades.index(decade_a)) | (1 << decades.index(decade_b))
            season_mask = (1 << seasons.index(season_a)) | (1 << seasons.index(season_b))
            masks.add((decade_mask, season_mask))
    return masks


mission_masks = {
    mission: two_scene_coverage_masks(strict_scenes.loc[strict_scenes["sensor"].eq(mission)])
    for mission in expected_missions
}
combined_masks = {(0, 0)}
for mission in expected_missions:
    combined_masks = {
        (decade_a | decade_b, season_a | season_b)
        for decade_a, season_a in combined_masks
        for decade_b, season_b in mission_masks[mission]
    }

complete_mask = ((1 << len(decades)) - 1, (1 << len(seasons)) - 1)
joint_design_feasible = complete_mask in combined_masks
count_feasible = mission_feasibility["mission_count_feasible"].all()
assert count_feasible and joint_design_feasible

design_result = pd.DataFrame(
    [
        {
            "design_check": "exactly two unique scenes per L5/L7/L8/L9",
            "result": bool(count_feasible),
            "interpretation": f"{target_unique_scenes} total scenes are feasible",
        },
        {
            "design_check": "all four decades and all four seasons represented jointly",
            "result": bool(joint_design_feasible),
            "interpretation": "feasible from aggregate strata; dates remain unselected",
        },
        {
            "design_check": "every counted scene covers all three core sectors",
            "result": True,
            "interpretation": "strict ROI-derived metadata condition",
        },
    ]
)
display(design_result)

## 5. Water-level definition and evidence gate

Water level is scene-and-sector specific. It must not be copied from an availability ROI or inferred from acquisition time alone. The workflow distinguishes two quantities:

1. **Still-water anomaly for P023 pilot bands**

   $\Delta z_{SWL}(t,s) = \Delta z_{MSL,GTSM}(y,s) + z_{tide,FES2022}(t,s) + z_{surge,GTSM}(t,s)$

   Here $z_{tide,FES2022}$ is the documented sum of the ocean and loading tide at the exact scene UTC and sector node, and $\Delta z_{MSL,GTSM}$ is the annual anomaly centred on 1991–2020. The three non-overlapping P023 bands are $\Delta z_{SWL}<0$ m, $0\leq\Delta z_{SWL}<0.2$ m, and $\Delta z_{SWL}\geq0.2$ m relative to local AMSL. There is no tide-only pilot band.

2. **Later ODN-referenced effective level for correction**

   $z_{effective,ODN}(t,s) = z_{reference,ODN}(s) + \Delta z_{SWL}(t,s) + z_{wave,selected}(t,s)$

   The ODN reference surface and selected wave term are later pilot-to-freeze inputs; they are not used to choose dates here. All components must have a documented common vertical-reference treatment. GTSM total water level must never be combined with separate FES tide, in accordance with P044.

In [ ]:
water_evidence_paths = {name: paths[name] for name in water_evidence_names}
water_evidence_present = {name: path.is_file() for name, path in water_evidence_paths.items()}
if any(water_evidence_present.values()) and not all(water_evidence_present.values()):
    raise FileNotFoundError(f"Water-level evidence is incomplete: {water_evidence_present}")
water_level_evidence_ready = all(water_evidence_present.values())
water_level_summary = None

if not water_level_evidence_ready:
    display(
        Markdown(
            "**PENDING ACCESS — stop here.** The complete scene table, summary and MSL merge audit "
            "do not exist. FES2022 and GTSM v3 access, exact nodes, UTC sampling, "
            "vertical-reference handling and provenance must be resolved before any "
            "pilot date or final water-level band is selected. No substitute values have been used."
        )
    )
else:
    water_levels = pd.read_csv(paths["scene_water_levels"])
    water_level_run_summary = json.loads(paths["water_level_summary"].read_text())
    msl_merge_audit = json.loads(paths["msl_merge_audit"].read_text())
    for output_name, identity in water_level_run_summary["output_identities"].items():
        recorded_path = Path(identity["path"])
        output_path = recorded_path if recorded_path.is_absolute() else REPO_ROOT / recorded_path
        if not output_path.is_file() or sha256_file(output_path) != identity["sha256"]:
            raise ValueError(f"Water-level output identity changed: {output_name}")
    required_water_columns = {
        "source_scene_id",
        "pilot_sector_id",
        "acquisition_time_utc",
        "fes_ocean_tide_m",
        "fes_loading_tide_m",
        *PILOT_STILL_WATER_COMPONENT_COLUMNS,
        "fes_node_id",
        "fes_node_longitude_deg",
        "fes_node_latitude_deg",
        "fes_node_distance_m",
        "gtsm_station_id",
        "gtsm_station_longitude_deg",
        "gtsm_station_latitude_deg",
        "gtsm_station_distance_m",
        "gtsm_sample_status",
        "gtsm_surge_archive_sha256",
        "gtsm_surge_netcdf_member",
        "fes_ocean_flag",
        "fes_loading_flag",
        "fes_product_version",
        "gtsm_product_version",
        "gtsm_msl_reference_period",
        "vertical_reference_status",
    }
    missing_water_columns = required_water_columns.difference(water_levels.columns)
    if missing_water_columns:
        raise ValueError(f"Water-level evidence is missing columns: {sorted(missing_water_columns)}")
    if water_levels.duplicated(["source_scene_id", "pilot_sector_id"]).any():
        raise ValueError("Water-level evidence has duplicate scene–sector rows")
    if not set(water_levels["pilot_sector_id"]).issubset(core_sector_ids):
        raise ValueError("Water-level evidence contains an unapproved core-sector ID")

    water_level_scene_ids = set(water_levels["source_scene_id"])
    if water_level_scene_ids != pool_ids:
        raise ValueError("Water-level scene IDs must exactly match the frozen evaluation pool")

    represented_sectors = water_levels.groupby("source_scene_id")["pilot_sector_id"].agg(set)
    incomplete_scenes = represented_sectors.loc[represented_sectors.map(lambda values: values != core_sector_ids)]
    if not incomplete_scenes.empty:
        raise ValueError(f"Every represented scene must include all core sectors: {incomplete_scenes.index[:5].tolist()}")

    exact_utc = pd.to_datetime(
        water_levels["acquisition_time_utc"], format="mixed", utc=True, errors="coerce"
    )
    manifest_utc = pd.to_datetime(
        water_levels["source_scene_id"].map(strict_scenes.set_index("source_scene_id")["acquisition_time_utc"]),
        format="mixed",
        utc=True,
        errors="coerce",
    )
    if exact_utc.isna().any() or manifest_utc.isna().any() or not np.array_equal(exact_utc.array.asi8, manifest_utc.array.asi8):
        raise ValueError("Water-level acquisition_time_utc must exactly match the availability manifest")

    numeric_provenance_columns = [
        "fes_ocean_tide_m",
        "fes_loading_tide_m",
        "fes_node_longitude_deg",
        "fes_node_latitude_deg",
        "fes_node_distance_m",
        "gtsm_station_longitude_deg",
        "gtsm_station_latitude_deg",
        "gtsm_station_distance_m",
    ]
    numeric_provenance = water_levels[numeric_provenance_columns].apply(pd.to_numeric, errors="coerce")
    if not np.isfinite(numeric_provenance.to_numpy(dtype=float)).all():
        raise ValueError("FES/GTSM component and node/station coordinates and distances must be finite")
    if (numeric_provenance[["fes_node_distance_m", "gtsm_station_distance_m"]] < 0).any().any():
        raise ValueError("FES/GTSM node or station distances must be non-negative")

    text_provenance_columns = [
        "fes_node_id",
        "gtsm_station_id",
        "gtsm_surge_archive_sha256",
        "gtsm_surge_netcdf_member",
        "fes_product_version",
        "gtsm_product_version",
        "gtsm_msl_reference_period",
        "vertical_reference_status",
    ]
    blank_provenance = water_levels[text_provenance_columns].fillna("").astype(str).apply(lambda column: column.str.strip().eq(""))
    if blank_provenance.any().any():
        raise ValueError("FES/GTSM IDs, product versions, MSL reference period and vertical-reference status must be nonblank")
    if not water_levels["gtsm_msl_reference_period"].astype(str).str.strip().eq("1991-2020").all():
        raise ValueError("GTSM mean-sea-level anomalies must use the recorded 1991-2020 reference period")
    if not water_levels["gtsm_sample_status"].eq("valid").all():
        raise ValueError("Final water-level evidence must contain only valid GTSM samples")
    if not water_levels[["fes_ocean_flag", "fes_loading_flag"]].eq(4).all().all():
        raise ValueError("Every FES component must carry the no-extrapolation flag 4")
    if not water_levels["gtsm_surge_archive_sha256"].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
        raise ValueError("Every GTSM surge row must retain a SHA-256 archive checksum")

    recorded_fes_tide = pd.to_numeric(water_levels["fes_astronomical_tide_m"], errors="coerce")
    water_levels = add_fes_astronomical_tide(water_levels)
    if not np.allclose(water_levels["fes_astronomical_tide_m"], recorded_fes_tide, rtol=0, atol=1e-6):
        raise ValueError("fes_astronomical_tide_m must equal ocean tide plus loading tide")

    recorded_still_water = pd.to_numeric(water_levels["pilot_still_water_anomaly_m"], errors="coerce")
    recorded_bands = water_levels["water_level_band"].copy()
    recorded_amsl_filter = water_levels["passes_local_amsl_filter"].copy()
    recorded_plus_0_2_filter = water_levels["passes_local_amsl_plus_0_2_m_filter"].copy()
    water_levels = add_pilot_still_water_anomaly(water_levels)
    if not np.allclose(water_levels["pilot_still_water_anomaly_m"], recorded_still_water, rtol=0, atol=1e-6):
        raise ValueError("Recorded still-water anomaly does not equal annual MSL anomaly plus FES tide plus GTSM surge")
    water_levels = add_p023_water_bands(water_levels)
    if not water_levels["water_level_band"].equals(recorded_bands):
        raise ValueError("Recorded P023 water-level bands do not match the frozen boundaries")
    if not water_levels["passes_local_amsl_filter"].equals(recorded_amsl_filter):
        raise ValueError("Recorded local-AMSL filter flags do not match the water-level bands")
    if not water_levels["passes_local_amsl_plus_0_2_m_filter"].equals(recorded_plus_0_2_filter):
        raise ValueError("Recorded AMSL+0.2 m filter flags do not match the water-level bands")

    expected_rows = len(evaluation_pool) * len(core_sector_ids)
    expected_status = "complete_water_level_evidence_dates_still_unselected"
    if water_level_run_summary["status"] != expected_status:
        raise ValueError("Water-level run summary does not record the completed evidence gate")
    if water_level_run_summary["candidate_scene_count"] != len(evaluation_pool) or water_level_run_summary["scene_sector_row_count"] != expected_rows:
        raise ValueError("Water-level run summary counts do not match the frozen pool")
    expected_merge_status = "complete_structural_and_descriptive_checks_no_overlap_test"
    if msl_merge_audit["status"] != expected_merge_status:
        raise ValueError("Annual-MSL historical/future merge evidence is incomplete")
    water_level_summary = (
        water_levels.groupby(["pilot_sector_id", "water_level_band"])
        .size()
        .rename("scene_count")
        .reset_index()
    )
    surge_evidence = pd.DataFrame([water_level_run_summary["surge_statistics"]["all_candidate_scene_sector_rows"]])
    merge_evidence = pd.DataFrame(msl_merge_audit["stations"])
    display(Markdown("Water-level evidence is present and numerically valid; dates are still unselected."))
    display(Markdown("**P023 scene counts by sector and water-level band**"))
    display(water_level_summary)
    display(Markdown("**P055 surge diagnostics across all evaluated scene–sector rows**"))
    display(surge_evidence)
    display(Markdown("**Annual-MSL historical/future boundary evidence**"))
    display(merge_evidence)
    display(Markdown("Sampling nodes remain provisional OS-seed geometry; these anomalies are not an ODN conversion or gauge validation."))

### Water-level coverage evidence

The figure below shows whether the frozen evaluation pool supplies candidates in each P023 band. It is a completeness diagnostic, not a scene ranking or a decision about which low-water filter to freeze.

In [ ]:
if water_level_evidence_ready:
    band_counts = (
        water_level_summary.pivot(
            index="pilot_sector_id", columns="water_level_band", values="scene_count"
        )
        .reindex(columns=P023_WATER_LEVEL_BANDS, fill_value=0)
        .fillna(0)
        .astype(int)
    )
    readable_sector_names = {
        sector_id: sector_id.replace("HOL_PILOT_", "").replace("_", " " ).title()
        for sector_id in band_counts.index
    }
    readable_band_names = {
        "below_local_amsl": "Below local AMSL",
        "local_amsl_to_below_plus_0_2_m": "AMSL to < AMSL + 0.2 m",
        "at_or_above_local_amsl_plus_0_2_m": "At or above AMSL + 0.2 m",
    }
    plot_counts = band_counts.rename(index=readable_sector_names, columns=readable_band_names)

    water_figure_path = REPO_ROOT / "outputs/figures/pilot-water-level-band-coverage.png"
    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    plot_counts.plot(
        kind="bar", stacked=True, ax=ax, color=["#4C78A8", "#F2CF5B", "#E45756"]
    )
    ax.set_title("Frozen Landsat evaluation pool by P023 water-level band")
    ax.set_xlabel("Core pilot sector")
    ax.set_ylabel("Scene count")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(
        title="Still-water anomaly band", frameon=False,
        loc="upper left", bbox_to_anchor=(1.01, 1),
    )
    fig.savefig(water_figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Wrote {water_figure_path.relative_to(REPO_ROOT)}")

## Final gate

There are intentionally no scene-ranking or shortlist-writing cells after the water-level check. A missing water-level file leaves selection pending. A valid file would permit a separate, explicit human review of a candidate shortlist; it would not authorise imagery retrieval or shoreline extraction.

In [ ]:
selection_gate = pd.DataFrame(
    [
        {
            "gate": "eight-scene metadata design",
            "status": "feasible" if joint_design_feasible else "not feasible",
        },
        {
            "gate": "FES2022 + GTSM v3 scene–sector evidence",
            "status": "present; human review still required" if water_level_evidence_ready else "pending access",
        },
        {
            "gate": "final pilot date shortlist",
            "status": "not selected or written",
        },
        {
            "gate": "imagery retrieval and shoreline extraction",
            "status": "not authorised",
        },
    ]
)
display(selection_gate)